In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os
import pickle
import urllib.request
import tarfile

# DATA PARAMETERS
BATCH_SIZE = 256
INPUT_SHAPE = (3, 32, 32)
NUM_CLASSES = 10
IMAGE_SIZE = 48
PATCH_SIZE = 6
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2

# OPTIMIZATION
LEARNING_RATE = 5e-4  # Adjusted learning rate
WEIGHT_DECAY = 5e-4
EPOCHS = 100

# ViT ARCHITECTURE
PROJECTION_DIM = 128
NUM_HEADS = 4
NUM_LAYERS = 4
MLP_UNITS = [PROJECTION_DIM * 2, PROJECTION_DIM]

# TOKENLEARNER
NUM_GROUPS = 4
NUM_TOKENS = 8

# Download and extract CIFAR-10 dataset
def download_cifar10():
    url = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
    filename = "cifar-10-python.tar.gz"
    folder = "cifar-10-batches-py"
    if not os.path.exists(folder):
        print("Downloading CIFAR-10 dataset...")
        urllib.request.urlretrieve(url, filename)
        with tarfile.open(filename, "r:gz") as tar:
            tar.extractall()
        os.remove(filename)

download_cifar10()

# Normalization parameters
mean = np.array([0.4914, 0.4822, 0.4465])
std = np.array([0.247, 0.243, 0.261])

# Custom Transformations
def preprocess_image(image):
    image = image.resize((IMAGE_SIZE, IMAGE_SIZE))
    image = np.array(image).transpose((2, 0, 1)) / 255.0
    image = (image - mean[:, None, None]) / std[:, None, None]  # Normalize
    return torch.tensor(image, dtype=torch.float32)

# Custom CIFAR-10 Dataset Class
class CIFAR10Dataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image = self.data[idx].transpose((1, 2, 0))  # Convert (C, H, W) to (H, W, C)
        image = Image.fromarray(image)
        image = preprocess_image(image)
        label = self.labels[idx]
        return image, label

# Load full CIFAR-10 dataset
train_data, train_labels = [], []
test_data, test_labels = None, None
for i in range(1, 6):
    with open(f"cifar-10-batches-py/data_batch_{i}", 'rb') as fo:
        batch = pickle.load(fo, encoding='bytes')
        train_data.append(batch[b'data'].reshape(-1, 3, 32, 32).astype(np.uint8))
        train_labels.extend(batch[b'labels'])
train_data = np.vstack(train_data)
train_labels = np.array(train_labels)

with open("cifar-10-batches-py/test_batch", 'rb') as fo:
    batch = pickle.load(fo, encoding='bytes')
    test_data = batch[b'data'].reshape(-1, 3, 32, 32).astype(np.uint8)
    test_labels = np.array(batch[b'labels'])

# Splitting Train and Validation Data
train_data, val_data = train_data[:45000], train_data[45000:]
train_labels, val_labels = train_labels[:45000], train_labels[45000:]

train_dataset = CIFAR10Dataset(train_data, train_labels)
val_dataset = CIFAR10Dataset(val_data, val_labels)
test_dataset = CIFAR10Dataset(test_data, test_labels)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Token Learner Module
class TokenLearner(nn.Module):
    def __init__(self, num_tokens=NUM_TOKENS):
        super().__init__()
        self.conv1 = nn.Conv2d(PROJECTION_DIM, PROJECTION_DIM, kernel_size=1, groups=NUM_GROUPS, bias=False)
        self.conv2 = nn.Conv2d(PROJECTION_DIM, num_tokens, kernel_size=1, bias=False)

    def forward(self, x):
        B, H, W, C = x.shape
        x = x.permute(0, 3, 1, 2)
        x = F.layer_norm(x, [C, H, W])
        attention_maps = self.conv1(x)
        attention_maps = self.conv2(attention_maps)
        attention_maps = F.softmax(attention_maps.view(B, NUM_TOKENS, -1), dim=-1)
        x = x.view(B, C, -1).permute(0, 2, 1)
        x = torch.einsum("bsi,bid->bsd", attention_maps, x)
        return x

# Vision Transformer Model
class VisionTransformer(nn.Module):
    def __init__(self, use_token_learner=True):
        super().__init__()
        self.patch_embedding = nn.Conv2d(3, PROJECTION_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        self.pos_embedding = nn.Parameter(torch.randn(1, NUM_PATCHES, PROJECTION_DIM))
        self.transformer_layers = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=PROJECTION_DIM, nhead=NUM_HEADS),
            num_layers=NUM_LAYERS,
        )
        self.token_learner = TokenLearner() if use_token_learner else None
        self.mlp_head = nn.Sequential(
            nn.LayerNorm(PROJECTION_DIM),
            nn.Linear(PROJECTION_DIM, NUM_CLASSES)
        )
    
    def forward(self, x):
        x = self.patch_embedding(x)
        x = x.flatten(2).transpose(1, 2)
        x = x + self.pos_embedding
        x = self.transformer_layers(x)
        if self.token_learner:
            x = x.view(x.shape[0], int(NUM_PATCHES ** 0.5), int(NUM_PATCHES ** 0.5), -1)
            x = self.token_learner(x)
        x = x.mean(dim=1)
        return self.mlp_head(x)

# Training Function
def train_model(model):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(EPOCHS):
        model.train()
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {loss.item():.4f}")
    
    test_accuracy(model, device)

def test_accuracy(model, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Test Accuracy: {100 * correct / total:.2f}%")
# Run Training
model = VisionTransformer(use_token_learner=True)
train_model(model)


/home/dyw7/.conda/envs/scGPT3/lib/python3.10/site-packages/torch/nn/modules/transformer.py:282: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Epoch 1/100, Loss: 1.9720
Epoch 2/100, Loss: 1.7550
Epoch 3/100, Loss: 1.4981
Epoch 4/100, Loss: 1.4447
Epoch 5/100, Loss: 1.2420
Epoch 6/100, Loss: 1.1708
Epoch 7/100, Loss: 1.1622
Epoch 8/100, Loss: 1.1157
Epoch 9/100, Loss: 1.2499
Epoch 10/100, Loss: 0.9382
Epoch 11/100, Loss: 0.8769
Epoch 12/100, Loss: 0.9071
Epoch 13/100, Loss: 1.0037
Epoch 14/100, Loss: 0.8644
Epoch 15/100, Loss: 0.8682
Epoch 16/100, Loss: 0.8545
Epoch 17/100, Loss: 0.7785
Epoch 18/100, Loss: 0.7400
Epoch 19/100, Loss: 0.8474
Epoch 20/100, Loss: 0.6608
Epoch 21/100, Loss: 0.6276
Epoch 22/100, Loss: 0.7519
Epoch 23/100, Loss: 0.6994
Epoch 24/100, Loss: 0.6461
Epoch 25/100, Loss: 0.5687
Epoch 26/100, Loss: 0.5851
Epoch 27/100, Loss: 0.5679
Epoch 28/100, Loss: 0.6373
Epoch 29/100, Loss: 0.4577
Epoch 30/100, Loss: 0.3958
Epoch 31/100, Loss: 0.3841
Epoch 32/100, Loss: 0.4252
Epoch 33/100, Loss: 0.3755
Epoch 34/100, Loss: 0.4127
Epoch 35/100, Loss: 0.4264
Epoch 36/100, Loss: 0.3209
Epoch 37/100, Loss: 0.2664
Epoch 38/1